# Elasticity model — derivation and worked examples

This notebook walks through the economic theory behind PriceSignal's pricing recommendations.

## 1. Price elasticity of demand

The price elasticity of demand (PED) measures how sensitive quantity demanded is to a change in price:

$$\varepsilon = \frac{\partial \ln Q}{\partial \ln P} = \frac{\% \Delta Q}{\% \Delta P}$$

- $\varepsilon < -1$: **Elastic** demand — a 1% price increase causes >1% drop in quantity.
- $-1 < \varepsilon < 0$: **Inelastic** demand — consumers are relatively price-insensitive.
- $\varepsilon = -1$: **Unit elastic** — revenue is maximised here.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

plt.style.use('dark_background')
plt.rcParams['figure.facecolor'] = '#141716'
plt.rcParams['axes.facecolor'] = '#1c1f1d'
plt.rcParams['axes.edgecolor'] = '#2a2e2b'
plt.rcParams['grid.color'] = '#2a2e2b'
print('Setup complete')

## 2. Simulating price / demand data

In [ ]:
np.random.seed(42)
TRUE_ELASTICITY = -1.8  # elastic demand

prices = np.linspace(10, 100, 40)
noise  = np.random.normal(0, 0.05, size=len(prices))
demand = 500 * (prices ** TRUE_ELASTICITY) * np.exp(noise)

df = pd.DataFrame({'price': prices, 'demand': demand})
df['log_price']  = np.log(df['price'])
df['log_demand'] = np.log(df['demand'])
df.head()

## 3. Log-log OLS regression

In [ ]:
model = LinearRegression().fit(
    df['log_price'].values.reshape(-1, 1),
    df['log_demand'].values
)
estimated_elasticity = model.coef_[0]
r2 = model.score(df['log_price'].values.reshape(-1, 1), df['log_demand'].values)

print(f'True elasticity:      {TRUE_ELASTICITY:.3f}')
print(f'Estimated elasticity: {estimated_elasticity:.3f}')
print(f'R²:                   {r2:.4f}')

## 4. Revenue-maximising price (Lerner markup)

Total revenue: $R = P \cdot Q(P)$. Setting $\frac{dR}{dP} = 0$ and solving gives:

$$P^* = \frac{\varepsilon}{\varepsilon + 1} \cdot MC$$

This is the **Lerner markup formula**. It only applies when $\varepsilon < -1$ (elastic demand). With inelastic demand, the profit-maximising strategy is to raise price.

In [ ]:
def revenue_maximising_price(elasticity: float, marginal_cost: float, margin_floor_pct: float = 10.0) -> float:
    if elasticity >= -1:
        return marginal_cost * (1 + margin_floor_pct / 100)  # inelastic: hold price up
    optimal = (elasticity / (elasticity + 1)) * marginal_cost
    floor   = marginal_cost * (1 + margin_floor_pct / 100)
    return max(optimal, floor)

MC = 30.0  # assumed marginal cost
P_star = revenue_maximising_price(estimated_elasticity, MC)
print(f'Marginal cost:            ${MC:.2f}')
print(f'Optimal price (P*):       ${P_star:.2f}')
print(f'Expected revenue at P*:   ${P_star * (500 * P_star**estimated_elasticity):.2f}')

## 5. Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: price vs demand
axes[0].scatter(df['price'], df['demand'], alpha=0.6, color='#22c55e', s=30)
axes[0].set_title('Price vs Demand', color='white')
axes[0].set_xlabel('Price ($)'); axes[0].set_ylabel('Quantity demanded')
axes[0].axvline(P_star, color='#f59e0b', linestyle='--', label=f'P* = ${P_star:.2f}')
axes[0].legend(fontsize=9)

# Panel 2: log-log with regression line
log_p_range = np.linspace(df['log_price'].min(), df['log_price'].max(), 100)
log_q_pred  = model.predict(log_p_range.reshape(-1, 1))
axes[1].scatter(df['log_price'], df['log_demand'], alpha=0.6, color='#22c55e', s=30)
axes[1].plot(log_p_range, log_q_pred, color='#3b82f6', linewidth=2, label=f'ε = {estimated_elasticity:.2f}')
axes[1].set_title(f'Log-Log OLS  (R²={r2:.3f})', color='white')
axes[1].set_xlabel('log(Price)'); axes[1].set_ylabel('log(Quantity)')
axes[1].legend(fontsize=9)

# Panel 3: revenue curve
p_range  = np.linspace(15, 120, 200)
q_range  = 500 * p_range ** estimated_elasticity
rev_range = p_range * q_range
axes[2].plot(p_range, rev_range, color='#22c55e', linewidth=2)
axes[2].axvline(P_star, color='#f59e0b', linestyle='--', label=f'P* = ${P_star:.2f}')
axes[2].set_title('Revenue curve', color='white')
axes[2].set_xlabel('Price ($)'); axes[2].set_ylabel('Total revenue ($)')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../docs/elasticity_model.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to docs/elasticity_model.png')